Imports


In [ ]:
import numpy as np
import functools, operator


In [ ]:
N_INPUTS: int = 3
N_LAYERS: int = 3
N_NODESPERLAYER: int = 8
N_OUTPUTS: int = 1

rng = np.random.default_rng(42)

inWeights = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_NODESPERLAYER, N_INPUTS])
inBias = rng.normal(0, np.sqrt(2 / (N_INPUTS)), N_NODESPERLAYER)
layerWeights = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_LAYERS - 1, N_NODESPERLAYER, N_NODESPERLAYER])
layerBias = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_LAYERS - 1, N_NODESPERLAYER])
outWeights = rng.normal(0, np.sqrt(2 / (N_INPUTS)), [N_OUTPUTS, N_NODESPERLAYER])
outBias = rng.normal(0, np.sqrt(2 / (N_INPUTS)))


def relu(x):
    return np.maximum(0, x)

# Forwarpass
def RunNetwork(_input, _inWeights, _inBias, _layerWeights, _layerBias, _outWeights, _outBias):
    act = relu(_inWeights @ _input + _inBias)  # input linear + relu
    for W, b in zip(_layerWeights, _layerBias):
        act = relu(W @ act + b)  # loop over layers: linear + relu
    act = _outWeights @ act + _outBias  # linear output layer
    return act


RunNetwork(rng.random(3), inWeights, inBias, layerWeights, layerBias, outWeights, outBias)

In [ ]:
class Tensor:
    def __init__(self, data: np.ndarray | list[float]):
        if type(data) == np.ndarray:
            self.data = data
        else:
            self.data = np.array(data, dtype=np.double)
        self.shape = np.shape(self.data)
        self.grad = np.zeros(self.shape)
        self._backward = lambda: None

    def __repr__(self):
        return f"Tensor(data=\n{self.data})"

    def reshape(self, newShape):
        self.data.reshape(newShape)
        self.shape = np.shape(self.data)

    # --- Rechenoperationen ---
    # Komponentenweise Addition
    def __add__(self, other):
        out = Tensor(self.data + other.data)

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    # Matrix-Matrix Multiplikation
    def __matmul__(self, other):
        out = Tensor(self.data @ other.data)

        def _backward():
            self.grad += 0
            other.grad += 0

        out._backward = _backward
        return out

    # Komponentenweise Multiplikation
    def __mul__(self, other):
        out = Tensor(self.data * other.data)

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out


def LinTens(inTensor: Tensor | list[Tensor], schema: str) -> Tensor:
    tensors = [inTensor] if isinstance(inTensor, Tensor) else inTensor  # Normalize input tensors to list
    out = Tensor(np.einsum(schema, *[t.data for t in tensors]))  # Forwardpass

    split_schema = schema.split("->")
    inind = split_schema[0].split(",")
    outind = split_schema[1]

    # Identify missing indeces, *and their position*, in target. Remove missing from target. Kontract what is left.
    # Add 1D axes at missing spots. Broadcast new axes to desired length.

    def _backward():
        for idt, t in enumerate(tensors):
            domain: list[str] = [chars for i, chars in enumerate(inind) if idt != i] + [outind]  # ordered list of index strings
            domain_set: set[str] = set("".join(domain))  # unique domain indeces

            missing: set[int] = {i for i, char in enumerate(outind) if char not in domain_set}  # positions of indeces in out_index, that arent in domain

            kept: str = "".join(char for i, char in enumerate(inind[idt]) if i not in missing)  # target without indeces, that dont occure in domain
            back_schema: str = ",".join(domain) + "->" + kept

            others: list[np.ndarray] = [tens.data for i, tens in enumerate(tensors) if idt != i]  # other tensors data

            g = np.einsum(back_schema, *others, out.grad)  # Contract the Tensor in all available indeces

            target = inind[idt]
            for pos, c in enumerate(target):
                if c in missing:
                    g = np.expand_dims(g, pos)

            g = np.broadcast_to(g, t.data.shape)
            t.grad += g

    out._backward = _backward
    return out

Contraction:
+ Map: "i->.", "i,i->." Jacobian: "i"
+ Map: "i,j->_" Jacobian: x: "i" mit x_i= y_1+y_2+...
+ Map: "ij,jk->ik" Jacobian: x: ij ik mit 
x_ijlk = \delta_il \delta_jk

X_lk = \sum_j=1^n A_lj B_jk, x_ijlk = \delta_il B_jk

1. Alle  auf maximale Größe bringen.




In [ ]:

x = np.arange(2)
y = np.expand_dims(x, (0,-1))
y = np.broadcast_to(y, (2,2,2))

print("shape x: \n", np.shape(x))
print("shape y: \n", np.shape(y))
print("y: \n", y)




def stretch(tensor: Tensor, insertion: tuple[int, ...], length: tuple[int, ...]) -> Tensor:
    array = np.expand_dims(tensor.data, insertion)

    shape = list(tensor.shape)
    for i, val in zip(insertion, length): 
        shape.insert(i, val)
    out = Tensor(np.broadcast_to(array, shape))
    return out

def insert(tensor: Tensor, insertion: tuple[int, ...]):
    return Tensor(np.expand_dims(tensor.data, insertion))



def contract(tensor: Tensor, axis: tuple[int, ...]):
    out = Tensor(np.sum(tensor.data, axis=axis)) # evtl: keepdims=True für 1 lange dimensionen
    return out

def tensMult(tensors: list[Tensor], schema: str):
    instructions = decode(schema)
    expand_instructions = instructions[0]
    contract_instructions = instructions[1]

    # expanded = functools.reduce(operator.mul, [insert(tensor, instr).data for tensor, instr in zip(tensors, expand_instructions)])

    expanded = [insert(tensor, instr).data for tensor, instr in zip(tensors, expand_instructions)]
    elem_prod = expanded[0] * 0 + 1
    for tensor in expanded:
        elem_prod = elem_prod * tensor

    return contract(Tensor(elem_prod), contract_instructions)



# Kann Indezes beliebig verschränken, nicht aber vertauschen "ij, ji -> ." macht das gleiche wie "ij, ij -> ."
def decode(schema: str):
    split = schema.replace(" ", "").replace(".", "").split("->")
    inList = split[0].split(",")
    outList = split[1]

    allchars = "".join(dict.fromkeys("".join(inList))) # remove duplicates while keeping order of firs appearances. join to single string

    stretchlist = [tuple(i for i, char in enumerate(allchars) if char not in string) for string in inList]
    contrlist = tuple(i for i, char in enumerate(allchars) if char not in outList)
    return (stretchlist, contrlist)

In [ ]:
# x = Tensor([1,1])
# y = Tensor([1, 10, 100])

x = Tensor(np.arange(9.0).reshape([3,3]))
y = Tensor(np.identity(3))
y.data[1,1] = -1.0
tensMult([x, y], "ij, jk -> ik")


# x = np.arange(9.0).reshape([3,3])
# np.einsum("ii->", x) 


In [ ]:
x = Tensor(np.arange(3.0))  # [0, 1, 2]

y = Tensor(np.array([1, 10, 100]))  # [1, 10, 100]

x = stretch(x, (0,), (3,))  # [[0, 0, 0], [1, 1, 1], [2, 2, 2]]

y = stretch(y, (0,), (3,))

z = contract(x * y, (0,))

z

In [ ]:
x = np.arange(9).reshape((3,3))

y = np